# Phase 6-7 学习指南：股票详情与观察名单

**目标：** 建立个股分析和多股票追踪的能力。

---
# Phase 6 — Stock Detail Page

构建个股的全方位分析视图。

## 1. 股票分析的核心维度

分析一只股票应该看什么？

| 维度 | 指标 | 目的 |
|------|------|------|
| **价格走势** | K线、均线 | 了解趋势 |
| **技术指标** | RSI, MACD, BB | 寻找信号 |
| **风险指标** | 波动率、回撤、Beta | 评估风险 |
| **相对表现** | vs SPY, vs 板块 | 对标基准 |
| **基本面** | PE, 收入增长 | 了解价值（可选）|

## 2. 相对表现分析

个股表现不能孤立看，要和基准对比。

**常用基准：**
- SPY（大盘基准）
- 板块 ETF（行业基准）

**计算方法：**
```
相对收益率 = 股票收益率 - 基准收益率
相对强度 = 股票归一化价格 / 基准归一化价格
```

**解读：**
- 相对强度上升 = 跑赢基准
- 相对强度下降 = 跑输基准

In [ ]:
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt

# 下载股票和基准数据
ticker = "AAPL"
benchmark = "SPY"

stock = yf.download(ticker, period="1y", progress=False)['Adj Close']
bench = yf.download(benchmark, period="1y", progress=False)['Adj Close']

# 对齐日期
data = pd.concat([stock, bench], axis=1, join='inner')
data.columns = ['Stock', 'Benchmark']

# 计算相对表现
data['Stock_Norm'] = data['Stock'] / data['Stock'].iloc[0] * 100
data['Bench_Norm'] = data['Benchmark'] / data['Benchmark'].iloc[0] * 100
data['Relative_Strength'] = data['Stock_Norm'] / data['Bench_Norm'] * 100

# 计算 Alpha（超额收益）
stock_return = (data['Stock'].iloc[-1] / data['Stock'].iloc[0] - 1) * 100
bench_return = (data['Benchmark'].iloc[-1] / data['Benchmark'].iloc[0] - 1) * 100
alpha = stock_return - bench_return

print(f"{ticker} 收益率: {stock_return:.2f}%")
print(f"{benchmark} 收益率: {bench_return:.2f}%")
print(f"Alpha (超额收益): {alpha:.2f}%")

# 可视化
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

# 价格对比
axes[0].plot(data.index, data['Stock_Norm'], label=ticker, linewidth=2)
axes[0].plot(data.index, data['Bench_Norm'], label=benchmark, linewidth=2, alpha=0.7)
axes[0].set_title(f'{ticker} vs {benchmark} (Normalized)')
axes[0].set_ylabel('Normalized Price')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 相对强度
axes[1].plot(data.index, data['Relative_Strength'], color='purple', linewidth=1.5)
axes[1].axhline(y=100, color='red', linestyle='--', alpha=0.5)
axes[1].fill_between(data.index, 100, data['Relative_Strength'], 
                      where=data['Relative_Strength']>100, alpha=0.3, color='green')
axes[1].fill_between(data.index, 100, data['Relative_Strength'], 
                      where=data['Relative_Strength']<100, alpha=0.3, color='red')
axes[1].set_title(f'Relative Strength: {ticker}/{benchmark}')
axes[1].set_ylabel('Relative Strength')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. 板块相对表现

股票还要和所在板块对比，判断是个股问题还是行业问题。

In [ ]:
# 股票 vs 板块
ticker = "NVDA"
sector_etf = "XLK"  # 科技板块

stock = yf.download(ticker, period="6m", progress=False)['Adj Close']
sector = yf.download(sector_etf, period="6m", progress=False)['Adj Close']
spy = yf.download("SPY", period="6m", progress=False)['Adj Close']

# 对齐
data = pd.concat([stock, sector, spy], axis=1, join='inner')
data.columns = ['Stock', 'Sector', 'Market']

# 归一化
data = data / data.iloc[0] * 100

# 计算 Alpha
vs_sector_alpha = (data['Stock'].iloc[-1] - data['Sector'].iloc[-1])
vs_market_alpha = (data['Stock'].iloc[-1] - data['Market'].iloc[-1])

print(f"{ticker} vs Sector (XLK): {'+' if vs_sector_alpha > 0 else ''}{vs_sector_alpha:.1f}%")
print(f"{ticker} vs Market (SPY): {'+' if vs_market_alpha > 0 else ''}{vs_market_alpha:.1f}%")

# 可视化
plt.figure(figsize=(12, 5))
plt.plot(data.index, data['Stock'], label=ticker, linewidth=2)
plt.plot(data.index, data['Sector'], label=f'Sector ({sector_etf})', linewidth=1.5)
plt.plot(data.index, data['Market'], label='Market (SPY)', linewidth=1.5, alpha=0.7)
plt.title(f'{ticker} vs Sector vs Market')
plt.ylabel('Normalized Price (Start=100)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 4. 股票详情仪表板设计

一个完整的股票详情页应该包含：

```
┌─────────────────────────────────────────────────────┐
│  Ticker Input: [____]  [Load]                       │
├─────────────────────────────────────────────────────┤
│  Summary Metrics                                     │
│  ┌──────────┬──────────┬──────────┬──────────┐      │
│  │ Price    │ 52W High │ 52W Low  │ Vol      │      │
│  │ $XXX.XX  │ $XXX.XX  │ $XXX.XX  │ XX.XM    │      │
│  └──────────┴──────────┴──────────┴──────────┘      │
├─────────────────────────────────────────────────────┤
│  Candlestick Chart (with MA20/50/200)               │
│  [_____________________________________________]    │
├─────────────────────────────────────────────────────┤
│  Technical Indicators                                │
│  ┌──────────┬──────────┬──────────┬──────────┐      │
│  │ RSI (14) │ MACD     │ BB Upper │ ATR (14) │      │
│  │ XX.X     │ X.XX     │ $XXX.XX  │ X.XX     │      │
│  └──────────┴──────────┴──────────┴──────────┘      │
├─────────────────────────────────────────────────────┤
│  Risk Metrics                                        │
│  ┌──────────┬──────────┬──────────┬──────────┐      │
│  │ Vol (Ann)│ Max DD   │ Sharpe   │ Beta     │      │
│  │ XX.X%    │ -XX.X%   │ X.XX     │ X.XX     │      │
│  └──────────┴──────────┴──────────┴──────────┘      │
├─────────────────────────────────────────────────────┤
│  Relative Performance                                │
│  vs SPY:  [+XX.X%]  vs Sector:  [+XX.X%]            │
└─────────────────────────────────────────────────────┘
```

## 5. 综合分析函数

把所有指标整合成一份报告。

In [ ]:
import numpy as np

def analyze_stock(ticker, period="1y"):
    """
    股票综合分析报告
    """
    # 下载数据
    stock = yf.download(ticker, period=period, progress=False)
    info = yf.Ticker(ticker).info
    
    # 基本价格信息
    current_price = stock['Adj Close'].iloc[-1]
    high_52w = stock['Adj Close'].rolling(252).max().iloc[-1]
    low_52w = stock['Adj Close'].rolling(252).min().iloc[-1]
    
    # 技术指标
    # MA
    ma20 = stock['Adj Close'].rolling(20).mean().iloc[-1]
    ma50 = stock['Adj Close'].rolling(50).mean().iloc[-1]
    ma200 = stock['Adj Close'].rolling(200).mean().iloc[-1] if len(stock) >= 200 else None
    
    # RSI
    delta = stock['Adj Close'].diff()
    gain = delta.where(delta > 0, 0).rolling(14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(14).mean()
    rs = gain / loss
    rsi = 100 - (100 / (1 + rs)).iloc[-1]
    
    # 风险指标
    returns = stock['Adj Close'].pct_change().dropna()
    ann_vol = returns.std() * np.sqrt(252)
    
    # 最大回撤
    cummax = stock['Adj Close'].cummax()
    drawdown = (stock['Adj Close'] - cummax) / cummax
    max_dd = drawdown.min()
    
    # 总收益
    total_return = (current_price / stock['Adj Close'].iloc[0] - 1) * 100
    
    # 打印报告
    print(f"\n{'='*50}")
    print(f"  {ticker} 分析报告")
    print(f"{'='*50}")
    
    print(f"\n【价格信息】")
    print(f"  当前价格: ${current_price:.2f}")
    print(f"  52周最高: ${high_52w:.2f} ({(current_price/high_52w-1)*100:.1f}% below)")
    print(f"  52周最低: ${low_52w:.2f} ({(current_price/low_52w-1)*100:.1f}% above)")
    
    print(f"\n【趋势判断】")
    print(f"  MA20: ${ma20:.2f} - {'Above ✅' if current_price > ma20 else 'Below ❌'}")
    print(f"  MA50: ${ma50:.2f} - {'Above ✅' if current_price > ma50 else 'Below ❌'}")
    if ma200:
        print(f"  MA200: ${ma200:.2f} - {'Above ✅' if current_price > ma200 else 'Below ❌'}")
    
    print(f"\n【技术指标】")
    print(f"  RSI(14): {rsi:.1f} - ", end="")
    if rsi > 70:
        print("超买 ⚠️")
    elif rsi < 30:
        print("超卖 ⚠️")
    else:
        print("中性")
    
    print(f"\n【风险指标】")
    print(f"  年化波动率: {ann_vol*100:.1f}%")
    print(f"  最大回撤: {max_dd*100:.1f}%")
    print(f"  期间收益: {total_return:.1f}%")
    
    print(f"\n{'='*50}\n")

# 测试
analyze_stock("AAPL")

## 自测问题（Phase 6）

1. 为什么个股表现要和 SPY 对比？

2. 一只股票跑赢 SPY 但跑输所在板块，说明了什么？

3. 相对强度指标从 110 降到 95，意味着什么？

4. 如果你要给别人展示一只股票的分析，你会展示哪些关键信息？

---
# Phase 7 — Watchlist System

管理多只股票的追踪和比较。

## 1. Watchlist 的核心功能

Watchlist 是一个股票观察名单，帮助你：
- 追踪多只股票的表现
- 快速比较不同股票
- 发现潜在机会

**关键功能：**
1. 添加/删除股票
2. 多股票数据同步更新
3. 多维度排名
4. 表现对比

## 2. 多时间框架收益

不同时间框架看不同的事情：

| 时间框架 | 用途 |
|----------|------|
| 5日 | 短期动量 |
| 20日 | 月度趋势 |
| 60日 | 季度表现 |
| YTD | 年度表现 |
| 1年 | 长期趋势 |

In [ ]:
def calculate_multi_period_returns(tickers):
    """
    计算多只股票在不同时间框架的收益率
    """
    results = []
    
    for ticker in tickers:
        try:
            df = yf.download(ticker, period="1y", progress=False)
            
            current = df['Adj Close'].iloc[-1]
            
            # 计算各期间收益
            r_5d = (current / df['Adj Close'].iloc[-5] - 1) * 100 if len(df) >= 5 else None
            r_20d = (current / df['Adj Close'].iloc[-20] - 1) * 100 if len(df) >= 20 else None
            r_60d = (current / df['Adj Close'].iloc[-60] - 1) * 100 if len(df) >= 60 else None
            
            # YTD
            ytd_start = df[df.index >= f"{df.index[-1].year}-01-01"]
            r_ytd = (current / ytd_start['Adj Close'].iloc[0] - 1) * 100 if len(ytd_start) > 0 else None
            
            # 1年
            r_1y = (current / df['Adj Close'].iloc[0] - 1) * 100
            
            results.append({
                'Ticker': ticker,
                '5D': r_5d,
                '20D': r_20d,
                '60D': r_60d,
                'YTD': r_ytd,
                '1Y': r_1y
            })
        except Exception as e:
            print(f"Error fetching {ticker}: {e}")
    
    return pd.DataFrame(results)

# 测试
watchlist = ["AAPL", "MSFT", "GOOGL", "AMZN", "NVDA", "META", "TSLA"]
returns_df = calculate_multi_period_returns(watchlist)

# 显示结果
print("\n多时间框架收益率 (%):")
print(returns_df.to_string(index=False))

# 排名（按 YTD）
print("\n按 YTD 收益排名:")
print(returns_df.sort_values('YTD', ascending=False)[['Ticker', 'YTD']].to_string(index=False))

## 3. 风险收益排名

不仅看收益，还要看风险。

In [ ]:
def watchlist_risk_return(tickers, period="1y"):
    """
    Watchlist 的风险收益分析
    """
    results = []
    
    for ticker in tickers:
        try:
            df = yf.download(ticker, period=period, progress=False)
            returns = df['Adj Close'].pct_change().dropna()
            
            # 收益
            total_return = (df['Adj Close'].iloc[-1] / df['Adj Close'].iloc[0] - 1) * 100
            
            # 波动率
            volatility = returns.std() * np.sqrt(252) * 100
            
            # 最大回撤
            cummax = df['Adj Close'].cummax()
            max_dd = ((df['Adj Close'] - cummax) / cummax).min() * 100
            
            # 夏普比率（假设无风险利率 5%）
            ann_return = (1 + total_return/100) ** (252/len(df)) - 1
            sharpe = (ann_return - 0.05) / (volatility/100)
            
            results.append({
                'Ticker': ticker,
                'Return': total_return,
                'Volatility': volatility,
                'MaxDD': max_dd,
                'Sharpe': sharpe
            })
        except Exception as e:
            print(f"Error: {ticker}")
    
    df = pd.DataFrame(results)
    
    # 添加排名
    df['Return_Rank'] = df['Return'].rank(ascending=False).astype(int)
    df['Risk_Rank'] = df['Volatility'].rank(ascending=True).astype(int)  # 波动低=好
    df['Sharpe_Rank'] = df['Sharpe'].rank(ascending=False).astype(int)
    
    return df

# 运行
risk_return_df = watchlist_risk_return(watchlist)

print("\n风险收益分析:")
print(risk_return_df.round(2).to_string(index=False))

## 4. 可视化对比

用散点图展示风险-收益关系。

In [ ]:
# 风险-收益散点图
plt.figure(figsize=(10, 6))

plt.scatter(risk_return_df['Volatility'], 
            risk_return_df['Return'], 
            s=100, alpha=0.7)

# 标注
for i, row in risk_return_df.iterrows():
    plt.annotate(row['Ticker'], 
                 (row['Volatility'], row['Return']),
                 xytext=(5, 5), textcoords='offset points')

# 参考线
plt.axhline(y=0, color='red', linestyle='--', alpha=0.3)
plt.axvline(x=risk_return_df['Volatility'].median(), color='gray', linestyle='--', alpha=0.3)

plt.xlabel('Volatility (%)')
plt.ylabel('Return (%)')
plt.title('Risk-Return Profile')
plt.grid(True, alpha=0.3)

# 标注象限
plt.text(plt.xlim()[1]*0.7, plt.ylim()[1]*0.9, '高风险高收益', fontsize=10, color='green')
plt.text(plt.xlim()[0]*1.1, plt.ylim()[1]*0.9, '低风险高收益\n(理想)', fontsize=10, color='blue')
plt.text(plt.xlim()[1]*0.7, plt.ylim()[0]*1.1, '高风险低收益\n(避免)', fontsize=10, color='red')
plt.text(plt.xlim()[0]*1.1, plt.ylim()[0]*1.1, '低风险低收益', fontsize=10, color='gray')

plt.tight_layout()
plt.show()

## 5. Watchlist 数据结构设计

```python
# 推荐的数据结构
watchlist = {
    "name": "Tech Stocks",
    "tickers": ["AAPL", "MSFT", "GOOGL", ...],
    "created_at": "2024-01-01",
    "last_updated": "2024-01-15",
    "notes": {
        "AAPL": "Waiting for earnings",
        "MSFT": "Strong cloud growth"
    }
}

# 存储为 JSON
import json
with open('data/watchlists/tech_stocks.json', 'w') as f:
    json.dump(watchlist, f, indent=2)
```

## 自测问题（Phase 7）

1. Watchlist 按收益排名和按夏普比率排名，结果可能不同，为什么？

2. 如果一只股票在「高风险低收益」象限，你会怎么处理？

3. 为什么要在 Watchlist 里加入备注功能？

4. 设计一个筛选条件，从 Watchlist 中找出「近期表现好且风险可控」的股票。

---
## 学习检查清单

### Phase 6
- [ ] 理解相对表现分析
- [ ] 会计算 Alpha（超额收益）
- [ ] 能对比股票 vs 板块 vs 市场
- [ ] 理解股票详情页的设计逻辑

### Phase 7
- [ ] 理解多时间框架收益的意义
- [ ] 能进行风险收益排名
- [ ] 会用散点图展示风险收益关系
- [ ] 理解 Watchlist 数据结构设计

---
**完成后，切换到 Codex 执行 Phase 6-7 的代码实现。**